# Меняем компоненты и обучаем модели в PyTorch / Diffusers

Откройте файл в новом Jupyter / Google Colab и выполняйте сверху вниз.
Первая ячейка устанавливает `diffusion-lab` из GitHub в текущий kernel.
Нужны Python 3.11+ и интернет для установки зависимостей.

Дальше обычные **Dataset → DataLoader → model → scheduler → optimizer → PyTorch loop → pipeline**.
Весь training loop виден в ноутбуке; loss — обычная функция, optimizer — `torch.optim`.
Каждый вариант действительно обучается и генерирует samples.

Меняем MLP / Transformer / свой модуль, MSE / Huber / свой loss, one-hot / embedding / Fourier,
optimizer, diffusion / flow, regularizers и задачи 1D / 3D / навигацию.
По умолчанию 2–3 эпохи на CPU; пара epsilon / v_prediction обучается по 20 эпох.
Это короткие учебные запуски: для исследований увеличивайте бюджет и повторяйте несколько seed.


In [ ]:
# Установить пакет из GitHub в Python текущего Jupyter kernel.
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "diffusion-lab[interactive] @ git+" + REPOSITORY_URL,
])


## Объекты и общие настройки

Все варианты лучей используют одинаковые данные и seed.
Baseline: `v_prediction` и cosine schedule, заканчивающийся почти Gaussian noise.
Число train timesteps, архитектуру, scheduler, loss callable и optimizer можно заменить независимо,
сохраняя совместимость target и sampler. Здесь нет обязательного trainer framework.


In [ ]:
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from IPython.display import Markdown, display
from diffusers import DDIMScheduler, FlowMatchEulerDiscreteScheduler
from diffusers.configuration_utils import register_to_config
from diffusers.utils.torch_utils import randn_tensor

from diffusion_lab import PointDenoiser, ResearchDiffusionPipeline
from diffusion_lab.environments import SampleDataset, make_environment
from diffusion_lab.conditioning import CONDITIONERS, register_conditioner
from diffusion_lab.losses import bounds, collision, endpoint, smoothness
from diffusion_lab.metrics import evaluate_samples
from diffusion_lab.experiment import endpoint_constraints
from diffusion_lab.visualization import plot_dataset, plot_trajectories

SEED, N_SAMPLES, EPOCHS = 42, 256, 3
DEVICE = "cpu"                         # "cuda" или "mps", если доступны
WIDTH, DEPTH, STEPS = 32, 1, 24
torch.set_num_threads(4)

rays = make_environment("rotating_rays", n_samples=N_SAMPLES, seed=SEED, n_classes=4)
base_model = {"hidden_dim": WIDTH, "num_layers": DEPTH}
base_scheduler = {"num_train_timesteps": 64, "prediction_type": "v_prediction",
                  "beta_schedule": "squaredcos_cap_v2", "clip_sample": False,
                  "timestep_spacing": "trailing"}
runs = {}


In [ ]:
def train_loop(model, scheduler, loader, optimizer, loss_fn, epochs=3,
               condition_dropout=0.15, regularizers=None, metadata=None,
               grad_clip=1.0, generator=None, initial_epoch=0, callback=None):
    """Plain PyTorch optimization; every object and loss callable can be replaced.

    Loss may be scalar or unreduced [B]. Regularizers are
    {name: (function(clean_prediction, conditions, metadata), weight)}.
    A CPU generator also works with CUDA/MPS through Diffusers' randn_tensor.
    """
    history = []
    flow = isinstance(scheduler, FlowMatchEulerDiscreteScheduler)
    for epoch in range(initial_epoch, initial_epoch + epochs):
        model.train()
        sums, count = {}, 0
        for batch in loader:
            clean = batch["samples"].to(device=model.device, dtype=model.dtype)
            conditions = batch.get("conditions")
            conditions = conditions.to(model.device) if conditions is not None else None
            noise = randn_tensor(clean.shape, generator=generator,
                                 device=model.device, dtype=model.dtype)
            if flow:
                sigma = torch.rand(len(clean), generator=generator).to(model.device)
                expanded_sigma = sigma.reshape(-1, *([1] * (clean.ndim - 1)))
                noisy = (1 - expanded_sigma) * clean + expanded_sigma * noise
                timesteps = sigma * scheduler.config.num_train_timesteps
                target = noise - clean                    # dx_sigma / d_sigma
            else:
                timesteps = torch.randint(scheduler.config.num_train_timesteps,
                                          (len(clean),), generator=generator).to(model.device)
                noisy = scheduler.add_noise(clean, noise, timesteps)
                prediction_type = scheduler.config.prediction_type
                if prediction_type == "epsilon":
                    target = noise
                elif prediction_type == "v_prediction":
                    target = scheduler.get_velocity(clean, noise, timesteps)
                elif prediction_type == "sample":
                    target = clean
                else:
                    raise ValueError(f"Unsupported prediction_type: {prediction_type}")

            keep = (torch.rand(len(clean), generator=generator) >= condition_dropout).to(model.device)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(noisy, timesteps, conditions=conditions, condition_mask=keep).sample
            regression = loss_fn(prediction, target).mean()
            loss = regression
            values = {"regression": regression.detach().item()}
            if regularizers:
                if flow:
                    clean_prediction = noisy - expanded_sigma * prediction
                    active = torch.ones(len(clean), dtype=torch.bool, device=model.device)
                else:
                    alpha = scheduler.alphas_cumprod.to(clean)[timesteps].reshape(
                        -1, *([1] * (clean.ndim - 1)))
                    clean_prediction = (
                        (noisy - (1 - alpha).sqrt() * prediction) / alpha.sqrt()
                        if prediction_type == "epsilon" else
                        alpha.sqrt() * noisy - (1 - alpha).sqrt() * prediction
                        if prediction_type == "v_prediction" else prediction
                    )
                    active = timesteps < scheduler.config.num_train_timesteps // 2
                if active.any():
                    selected_conditions = conditions[active] if conditions is not None else None
                    for name, (penalty_fn, weight) in regularizers.items():
                        penalty = penalty_fn(clean_prediction[active], selected_conditions, metadata or {})
                        loss = loss + weight * penalty
                        values[name] = penalty.detach().item()
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip, error_if_nonfinite=True)
            optimizer.step()
            values["loss"] = loss.detach().item()
            for name, value in values.items():
                sums[name] = sums.get(name, 0.0) + len(clean) * value
            count += len(clean)
        row = {name: value / count for name, value in sums.items()}
        row.update(epoch=epoch + 1, step=(epoch + 1) * len(loader))
        history.append(row)
        if callback is not None:
            callback(row)
    return history


In [ ]:
def train_case(name, *, dataset=rays, environment="rotating_rays", model=None,
               scheduler=None, loss_fn=None, epochs=EPOCHS, regularizers=None,
               n_samples=N_SAMPLES, environment_kwargs=None,
               optimizer_factory=torch.optim.AdamW, learning_rate=2e-3):
    """Удобство для сравнений: внутри те же обычные объекты и явный loop."""
    torch.manual_seed(SEED)
    if dataset is None:
        dataset = make_environment(environment, n_samples=n_samples, seed=SEED,
                                   **(environment_kwargs or {}))
    model_kwargs = {"sample_shape": dataset.sample_shape, **base_model,
                    "conditioner": {"categorical": "embedding", "continuous": "mlp", "none": "none"}[
                        dataset.condition_kind],
                    "num_classes": dataset.num_classes, "condition_dim": dataset.condition_dim}
    model = model if isinstance(model, nn.Module) else PointDenoiser(**{**model_kwargs, **(model or {})})
    scheduler = scheduler if scheduler is not None else DDIMScheduler(**base_scheduler)
    pipeline = ResearchDiffusionPipeline(model, scheduler).to(DEVICE)
    loader_generator = torch.Generator().manual_seed(SEED + 1)
    loader = DataLoader(dataset, batch_size=64, shuffle=True, generator=loader_generator)
    optimizer = optimizer_factory(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    train_generator = torch.Generator().manual_seed(SEED + 2)
    history = train_loop(model, scheduler, loader, optimizer,
                         nn.MSELoss() if loss_fn is None else loss_fn, epochs=epochs,
                         regularizers=regularizers, metadata=dataset.metadata,
                         generator=train_generator)
    indices = torch.randint(len(dataset), (64,), generator=torch.Generator().manual_seed(SEED + 3))
    conditions = dataset.conditions[indices] if dataset.conditions is not None else None
    constraints = endpoint_constraints(model, conditions) if hasattr(dataset, "environment") else {}
    output = pipeline(batch_size=64, conditions=conditions, num_inference_steps=STEPS,
                      generator=torch.Generator().manual_seed(SEED + 3), **constraints)
    result = {"dataset": dataset, "model": model, "scheduler": scheduler,
              "pipeline": pipeline, "loader": loader, "optimizer": optimizer,
              "train_generator": train_generator, "loader_generator": loader_generator,
              "history": history, "conditions": conditions, "output": output,
              "metrics": evaluate_samples(dataset, output.samples, conditions, seed=SEED)}
    runs[name] = result
    print(f"{name}: epochs={len(history)}, optimizer steps={history[-1]['step']}, "
          f"loss={history[-1]['loss']:.5f}")
    return result


def compare_points(names):
    fig, axes = plt.subplots(1, len(names) + 1, figsize=(4 * (len(names) + 1), 4),
                             constrained_layout=True)
    reference = runs[names[0]]["dataset"]
    plot_dataset(reference.samples, reference.labels, ax=axes[0], title="Reference")
    rows = ["| Вариант | Последний loss | Sliced Wasserstein ↓ |", "|---|---:|---:|"]
    for ax, name in zip(axes[1:], names):
        result = runs[name]
        labels = result["conditions"] if result["dataset"].condition_kind == "categorical" else None
        plot_dataset(result["output"].samples, labels, ax=ax, title=name)
        rows.append(f"| {name} | {result['history'][-1]['loss']:.4f} | "
                    f"{result['metrics']['sliced_wasserstein']:.4f} |")
    plt.show()
    plt.close()
    display(Markdown("\n".join(rows)))


## 1. Baseline: явная сборка и обычное обучение

Здесь создаём каждый объект отдельно. `train_loop` выше использует `zero_grad → backward → step`;
вы можете редактировать его тело, заменить optimizer или добавить learning rate scheduler.
Pipeline работает с уже обученной моделью; скрытой EMA-копии здесь нет.


In [ ]:
torch.manual_seed(SEED)
dataset = rays
loader_generator = torch.Generator().manual_seed(SEED + 1)
loader = DataLoader(dataset, batch_size=64, shuffle=True, generator=loader_generator)
model = PointDenoiser(sample_shape=dataset.sample_shape, **base_model,
                      conditioner="embedding", num_classes=dataset.num_classes)
scheduler = DDIMScheduler(**base_scheduler)
pipeline = ResearchDiffusionPipeline(model=model, scheduler=scheduler).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()
train_generator = torch.Generator().manual_seed(SEED + 2)

history = train_loop(model, scheduler, loader, optimizer, loss_fn, epochs=EPOCHS,
                     generator=train_generator)
indices = torch.randint(len(dataset), (64,), generator=torch.Generator().manual_seed(SEED + 3))
sample_conditions = dataset.conditions[indices]
generated = pipeline(batch_size=64, conditions=sample_conditions, num_inference_steps=STEPS,
                      generator=torch.Generator().manual_seed(SEED + 3))
runs["MLP"] = {"dataset": dataset, "model": model, "scheduler": scheduler, "pipeline": pipeline,
               "loader": loader, "optimizer": optimizer, "train_generator": train_generator,
               "loader_generator": loader_generator, "history": history,
               "conditions": sample_conditions, "output": generated,
               "metrics": evaluate_samples(dataset, generated.samples, sample_conditions)}
print("Actual optimizer updates:", history[-1]["step"])
plot_dataset(generated.samples, sample_conditions, title="MLP: после обучения")
plt.show()
plt.close()
print(runs["MLP"]["metrics"])


## 2. Меняем архитектуру: Transformer и свой модуль

`hidden_dim`, `num_layers`, `num_heads`, `backbone` — параметры `PointDenoiser`.
Для точек Transformer видит один токен; для траекторий `[B,L,D]` attention связывает L позиций.
Собственная модель — обычный PyTorch-модуль с Diffusers `ModelMixin`.
Добавим обучаемый residual input adapter, сохранив noisy sample / timestep / conditions и форму выхода.


In [ ]:
transformer = train_case("Transformer", model={"backbone": "transformer", "num_heads": 4})
compare_points(["MLP", "Transformer"])


In [ ]:
class AdapterDenoiser(PointDenoiser):
    @register_to_config
    def __init__(self, sample_shape=(2,), hidden_dim=32, num_layers=1, num_heads=4,
                 backbone="mlp", conditioner="embedding", num_classes=4,
                 condition_dim=0, adapter_scale=0.1):
        super().__init__(sample_shape=sample_shape, hidden_dim=hidden_dim, num_layers=num_layers,
                         num_heads=num_heads, backbone=backbone, conditioner=conditioner,
                         num_classes=num_classes, condition_dim=condition_dim)
        if len(sample_shape) != 1:
            raise ValueError("This example adapter handles points [B,D]")
        self.adapter = nn.Sequential(nn.Linear(sample_shape[0], hidden_dim), nn.SiLU(),
                                     nn.Linear(hidden_dim, sample_shape[0]))
        self.adapter_scale = adapter_scale

    def forward(self, sample, timestep, **kwargs):
        adapted = sample + self.adapter_scale * self.adapter(sample)
        return super().forward(adapted, timestep, **kwargs)


torch.manual_seed(SEED)             # Воспроизводимая инициализация собственного модуля
custom_model = AdapterDenoiser(sample_shape=rays.sample_shape, num_classes=rays.num_classes,
                               hidden_dim=WIDTH, adapter_scale=0.1)
custom_model_run = train_case("Custom model", model=custom_model)
compare_points(["MLP", "Custom model"])


## 3. Меняем loss: обычные PyTorch callables

`loss_fn` можно заменить на `nn.SmoothL1Loss()` или свою функцию без registry.
Она может вернуть scalar или `[B]`; loop применит `.mean()`.
У MSE, Huber и Charbonnier разные единицы, поэтому сравниваем также общую метрику генерации.


In [ ]:
def notebook_charbonnier(prediction, target):
    return ((prediction - target).square() + 1e-6).sqrt().flatten(1).mean(1)

huber_run = train_case("Huber", loss_fn=nn.SmoothL1Loss())
charbonnier_run = train_case("Charbonnier", loss_fn=notebook_charbonnier)
compare_points(["MLP", "Huber", "Charbonnier"])


## 4. Категориальный conditioner: one-hot ↔ embedding

Исходные labels остаются целочисленными ID `[B]`. Меняется только способ их кодирования.
Число классов автоматически берётся из датасета; one-hot не нужно строить в ноутбуке.


In [ ]:
one_hot_run = train_case("One-hot", model={"conditioner": "one_hot"})
compare_points(["One-hot", "MLP"])


## 5. Свои данные и непрерывный conditioner: MLP ↔ Fourier

Строим точки `(c, c²)` с шумом; условие `c` имеет форму `[N,1]`.
Собственный encoder принимает `(hidden_dim,num_classes,condition_dim)` и возвращает `[B,hidden_dim]`.
Для изображений / текстов / наборов признаков внутри него можно использовать более сложный encoder,
свернув его выход в вектор context. Null-condition и classifier-free dropout уже поддержаны denoiser.


In [ ]:
generator = torch.Generator().manual_seed(SEED)
context = torch.linspace(-1, 1, N_SAMPLES)[:, None]
points = torch.cat((context, context.square()), dim=1)
points += 0.02 * torch.randn(points.shape, generator=generator)
continuous_data = SampleDataset(points, context, metadata={"name": "conditioned_parabola"})

class FourierConditioner(nn.Module):
    def __init__(self, hidden_dim, num_classes, condition_dim):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(3 * condition_dim, hidden_dim), nn.SiLU(),
                                        nn.Linear(hidden_dim, hidden_dim))

    def forward(self, conditions):
        c = conditions.to(self.projection[0].weight.dtype)
        features = torch.cat((c, (torch.pi * c).sin(), (torch.pi * c).cos()), -1)
        return self.projection(features)

# Rerunning an edited class updates the factory for subsequently constructed models.
CONDITIONERS.pop("notebook_fourier", None)
register_conditioner("notebook_fourier")(FourierConditioner)

continuous_mlp = train_case("Continuous MLP", dataset=continuous_data,
                           model={"conditioner": "mlp"})
continuous_fourier = train_case("Continuous Fourier", dataset=continuous_data,
                               model={"conditioner": "notebook_fourier"})
compare_points(["Continuous MLP", "Continuous Fourier"])


In [ ]:
# Конкретное continuous условие: pipeline расширяет [K] до [B,K].
fixed_context = continuous_fourier["pipeline"](
    batch_size=64, conditions=torch.tensor([0.5]), num_inference_steps=STEPS,
    generator=torch.Generator().manual_seed(SEED + 3))
plot_dataset(fixed_context.samples, title="c=0.5: цель около (0.5, 0.25)")
plt.scatter([0.5], [0.25], marker="*", s=180, color="#e11d48", label="Target")
plt.legend()
plt.show()
plt.close()


## 6. Меняем target и метод обучения

В loop явно показаны targets: `noise`, `clean`, `scheduler.get_velocity(...)`.
Для rectified flow: `x_sigma=(1-sigma)*clean+sigma*noise`, `target=noise-clean`.
Flow использует `FlowMatchEulerDiscreteScheduler`, а не DDIM.

Сравнение epsilon и v получает одинаковый linear schedule с 1000 timesteps и по 20 эпох:
terminal noise почти N(0,I). Epsilon всё ещё может давать плохие samples — это покажут метрика и график.


In [ ]:
flow_run = train_case("Rectified flow", scheduler=FlowMatchEulerDiscreteScheduler(num_train_timesteps=64))
compare_points(["MLP", "Rectified flow"])

for prediction_type in ("epsilon", "v_prediction"):
    target_scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear",
                                    prediction_type=prediction_type, clip_sample=False,
                                    timestep_spacing="trailing")
    train_case(f"{prediction_type} (linear 1000)", scheduler=target_scheduler, epochs=max(20, EPOCHS))
compare_points(["epsilon (linear 1000)", "v_prediction (linear 1000)"])


## 7. Свой regularizer и замена optimizer

В loop видно, где regression loss дополняется scalar penalty на восстановленном x₀.
Для diffusion geometry применяется только на меньших уровнях шума.
Regularizers передаются как `{name: (callable, weight)}` без обязательной регистрации.
Обычный `torch.optim.AdamW` можно заменить на Adam, SGD или свой optimizer.


In [ ]:
def notebook_radius(clean_prediction, conditions, metadata):
    return torch.relu(clean_prediction.norm(dim=-1) - metadata.get("radius", 1.0)).square().mean()

regularized = train_case("Radius penalty", regularizers={"radius": (notebook_radius, 0.1)})
adam_run = train_case("Adam", optimizer_factory=torch.optim.Adam)
compare_points(["MLP", "Radius penalty", "Adam"])
print("Loss components:", regularized["history"][-1])


## 8. Меняем задачу и размерность — с обучением

Датасет задаёт `sample_shape`, `condition_dim` и `num_classes`, они передаются в модель.
Ниже 1D Gaussian mixture и 3D helix проходят реальный torch loop и генерацию.
Другие варианты сред: `n_moons`, `rings`, `spirals`, `blobs_3d`.


In [ ]:
for task in ("gaussian_mixture_1d", "helix_3d"):
    result = train_case(task, dataset=None, environment=task)
    plot_dataset(result["output"].samples, result["conditions"], title=f"{task}: {EPOCHS} epochs")
    plt.show()
    plt.close()
    print(task, "shape:", result["dataset"].sample_shape, "metrics:", result["metrics"])


## 9. Навигация частицы A → B со стенами — тоже обучение

Reference paths строит A*. Условие включает start, goal и walls. Модель получает весь путь
`[B,L,D]`; используем Transformer и penalties на smoothness, столкновения, endpoints и bounds.
Здесь только 32 примера и 2 эпохи: модель ещё может пересекать стены.
Endpoint constraint фиксирует A и B при генерации; нулевая endpoint error получается за счёт
этого ограничения, поэтому смотрим отдельно `collision_free_fraction`.


In [ ]:
navigation = train_case(
    "Navigation", dataset=None, environment="navigation_2d", n_samples=32,
    environment_kwargs={"path_length": 16, "grid_resolution": 17},
    model={"backbone": "transformer"}, loss_fn=nn.SmoothL1Loss(), epochs=2,
    regularizers={"smoothness": (smoothness, 0.05), "collision": (collision, 0.2),
                  "endpoint": (endpoint, 0.1), "bounds": (bounds, 0.1)},
)
nav_dataset, nav_model, nav_pipeline = navigation["dataset"], navigation["model"], navigation["pipeline"]
nav_conditions = nav_dataset.conditions[:1].expand(8, -1)
# Endpoint constraint: известные A/B зашумляются и закрепляются на каждом scheduler step.
constraints = endpoint_constraints(nav_model, nav_conditions)
nav_output = nav_pipeline(batch_size=8, conditions=nav_conditions, num_inference_steps=STEPS,
                          generator=torch.Generator().manual_seed(SEED + 3), **constraints)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
plot_trajectories(nav_dataset.samples[:1], walls=nav_dataset.metadata["walls"],
                  ax=axes[0], title="Reference A* route")
plot_trajectories(nav_output.samples, walls=nav_dataset.metadata["walls"],
                  ax=axes[1], title="Learned trajectories: 2 epochs")
plt.show()
plt.close()
print(evaluate_samples(nav_dataset, nav_output.samples, nav_conditions))


## 10. После обучения: CFG и число шагов

Это параметры обычного Diffusers pipeline call. Модель переобучать не нужно.
CFG использует null-condition, обученный через `condition_dropout`.
Держим одинаковые noise seed и condition; `0` — null-condition, `1` — обычное условие,
`>1` усиливает guidance. Усиление guidance не гарантирует улучшения качества.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, guidance in zip(axes, (0.0, 1.0, 2.0)):
    guided = pipeline(batch_size=64, conditions=torch.ones(64, dtype=torch.long),
                      num_inference_steps=STEPS, guidance_scale=guidance,
                      generator=torch.Generator().manual_seed(SEED + 3))
    plot_dataset(guided.samples, torch.ones(64, dtype=torch.long), ax=ax, title=f"condition=1, CFG={guidance}")
plt.show()
plt.close()

for steps in (8, 24):
    sampled = pipeline(batch_size=64, conditions=sample_conditions, num_inference_steps=steps,
                       generator=torch.Generator().manual_seed(SEED + 3))
    print("Steps:", steps, "SW:", evaluate_samples(dataset, sampled.samples, sample_conditions)["sliced_wasserstein"])


## 11. Продолжение обучения и сохранение в Diffusers / PyTorch

Для продолжения используем ту же модель, loader, optimizer и генератор noise.
Сохраняем pipeline через `save_pretrained`, состояние optimizer и RNG — через `torch.save`.
Это обычные форматы Diffusers / PyTorch. Архитектура и conditioner сохраняются в model config;
код пользовательских классов хранится вместе с исследовательским ноутбуком.


In [ ]:
def on_epoch(row):
    print(f"Epoch {row['epoch']}: loss={row['loss']:.4f}, optimizer steps={row['step']}")

extra_history = train_loop(model, scheduler, loader, optimizer, loss_fn, epochs=2,
                           generator=train_generator, initial_epoch=len(history), callback=on_epoch)
history.extend(extra_history)
print("Всего эпох:", len(history))
fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
for name in ("MLP", "Transformer", "One-hot"):
    rows = runs[name]["history"]
    ax.plot([row["epoch"] for row in rows], [row["loss"] for row in rows], marker="o", label=name)
ax.set(xlabel="Epoch", ylabel="Diffusion v_prediction MSE", title="Training loss")
ax.legend(frameon=False)
plt.show()
plt.close()


In [ ]:
from pathlib import Path

checkpoint_dir = Path("outputs/component_demo_checkpoint")
pipeline.save_pretrained(checkpoint_dir / "pipeline", safe_serialization=True)
torch.save({"optimizer": optimizer.state_dict(), "history": history,
            "train_generator": train_generator.get_state(),
            "loader_generator": loader_generator.get_state(),
            "dataset_samples": dataset.samples, "dataset_conditions": dataset.conditions},
           checkpoint_dir / "training.pt")

restored_pipeline = ResearchDiffusionPipeline.from_pretrained(
    checkpoint_dir / "pipeline", local_files_only=True, low_cpu_mem_usage=False).to(DEVICE)
state = torch.load(checkpoint_dir / "training.pt", map_location="cpu", weights_only=True)
restored_optimizer = torch.optim.AdamW(restored_pipeline.model.parameters(), lr=2e-3, weight_decay=1e-4)
restored_optimizer.load_state_dict(state["optimizer"])
restored_dataset = SampleDataset(state["dataset_samples"], state["dataset_conditions"])
restored_loader_generator = torch.Generator().set_state(state["loader_generator"])
restored_loader = DataLoader(restored_dataset, batch_size=64, shuffle=True,
                             generator=restored_loader_generator)
restored_train_generator = torch.Generator().set_state(state["train_generator"])
resumed_history = train_loop(restored_pipeline.model, restored_pipeline.scheduler, restored_loader,
                             restored_optimizer, loss_fn, epochs=1,
                             initial_epoch=len(state["history"]), generator=restored_train_generator)
print("После reload + обучения:", resumed_history[-1])
resumed_output = restored_pipeline(batch_size=64, conditions=sample_conditions, num_inference_steps=STEPS,
                                  generator=torch.Generator().manual_seed(SEED + 3))
plot_dataset(resumed_output.samples, sample_conditions, title="Reloaded + trained")
plt.show()
plt.close()


Собственный conditioner зарегистрируйте до `from_pretrained`. Если custom model определена только
в ноутбуке, сохраните код класса и сначала загрузите его через `AdapterDenoiser.from_pretrained(path / "model")`,
затем передайте этот экземпляр в `ResearchDiffusionPipeline.from_pretrained(path, model=your_model)`.

Теперь можно редактировать train loop, добавлять loss terms, `torch.optim.lr_scheduler`, AMP,
свою модель и собственные данные. Визуализации полей и потенциалов показаны в
`03_standalone_quickstart.ipynb`.

Для компактных запусков в пакете остаются опциональные `DiffusionTrainer` и `create_experiment`;
этот ноутбук использует стандартные объекты PyTorch / Diffusers напрямую.


## Интерактивные графики
Condition, timestep, CFG и тип поля пересчитываются по обученной модели.
Generate запускает один sampling; Play/Frame показывают его реальные состояния.


In [ ]:
from IPython.display import display
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
from diffusion_lab.visualization import notebook_field_explorer, notebook_sampling_explorer
field_controls = notebook_field_explorer(cases["MLP"]["pipeline"], rays, resolution=24)
sample_controls = notebook_sampling_explorer(cases["MLP"]["pipeline"], rays, steps=24)
display(field_controls, sample_controls)
